# Notebook 1 — Data audit and 15 URL features

**Project:** Lightweight and interpretable phishing website detection under temporal change.

Use PhreshPhish v1.0.1, official `phreshphish/phreshphish` repository.
Dataset: https://huggingface.co/datasets/phreshphish/phreshphish
Paper: Dalton et al., PhreshPhish, https://arxiv.org/abs/2507.10854 (v2, 2026).

This notebook audits metadata and extracts U01–U15. No models are fitted.
Official-test metadata may be checked for integrity; official-test model performance
is embargoed until Notebook 4. No feature selection uses official-test data.

Only `url` supplies predictors here. `html` is read in Notebook 2. `label` is the
outcome; `date`, `sha256`, row IDs and parsing diagnostics are never predictors.
Stored URLs must never be visited; stored HTML must never be executed.

Run SMOKE first, then FULL. Smoke outputs are not research results.
Outputs: compact Parquet tables, audit CSVs, an Excel summary, seven PNG figures,
feature definitions, package/PSL provenance, and a completion manifest.

The internal date cutoff is predeclared here so Notebook 2 can make H08 decisions
using earlier development rows only. Notebook 3 verifies and uses that same split.
Record permitted AI assistance and verified ethics status under UCB guidance.


In [ ]:
# Run in a fresh Kaggle Python session, before importing these packages.
import sys, subprocess, os
PINS = ["datasets==4.4.1", "huggingface-hub==0.36.0",
        "pyarrow==21.0.0", "tldextract==5.3.0", "xlsxwriter==3.2.5"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *PINS])
os.environ["HF_HOME"] = "/tmp/phreshphish_hf_cache"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "60"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"
import json, math, time, hashlib, platform, re, ipaddress, shutil
from pathlib import Path
from collections import Counter
from importlib import metadata, resources
from urllib.parse import urlsplit, parse_qsl
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import tldextract
from datasets import load_dataset
from huggingface_hub import HfApi, HfFileSystem
from IPython.display import display
SEED = 42
np.random.seed(SEED)
plt.rcParams.update({"figure.dpi":110, "savefig.dpi":300,
    "font.family":"DejaVu Sans", "font.size":10,
    "axes.spines.top":False, "axes.spines.right":False})
COLORS = {0:"#0072B2", 1:"#D55E00"}
versions = {p:metadata.version(p) for p in
    ["datasets","huggingface-hub","pyarrow","tldextract","numpy",
     "pandas","matplotlib","xlsxwriter","fsspec"]}
for spec in PINS:
    name, expected = spec.split("==")
    assert metadata.version(name) == expected
print("Python:", platform.python_version())
display(pd.Series(versions, name="version").to_frame())


In [ ]:
DATASET_ID = "phreshphish/phreshphish"
DATASET_VERSION = "1.0.1"
# Reviewed dataset commit. Do not replace this with 'main'.
REVISION = "eabec4b7a66324b79cc8a0ad856d1731dc26fe1a"
RUN_MODE = "SMOKE"                  # Change ONLY to "FULL" after smoke succeeds.
SMOKE_ROWS = 2000                   # First rows, for debugging, not inference.
BATCH_SIZE = 2048
assert RUN_MODE in {"SMOKE", "FULL"}
ROOT = Path("/kaggle/working/phishing_thesis") / RUN_MODE.lower()
for folder in ["features","metadata","tables","figures","manifests","logs"]:
    (ROOT/folder).mkdir(parents=True, exist_ok=True)
# Invalidate a previous completion marker before a new run.
(ROOT/"manifests/manifest_nb01.json").unlink(missing_ok=True)
EXPECTED_FIELDS = ["sha256","url","label","target","date","lang","lang_score","html"]
READ_COLUMNS = ["sha256","url","label","date"]
REPORTED_ROWS = {"train":498255, "test":168060}
LABEL_MAP = {"benign":0, "phish":1}
RUNTIMES, BLOCKERS = [], []
def save_json(relative, obj):
    path = ROOT/relative
    path.write_text(json.dumps(obj, indent=2, default=str), encoding="utf-8")
def sha_file(path):
    h = hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()
def save_figure(name, fig):
    fig.tight_layout()
    fig.savefig(ROOT/"figures"/name, bbox_inches="tight")
    plt.show()
    plt.close(fig)
def metadata_path(split):
    return ROOT/"metadata"/f"{split}.parquet"
def feature_path(split):
    return ROOT/"features"/f"url_features_{split}.parquet"
print("Mode:", RUN_MODE, "| Output:", ROOT)
print("Free working disk (GiB):", round(shutil.disk_usage(ROOT).free/2**30,2))
print("No HTML download in Notebook 1; no GPU required.")


In [ ]:
api = HfApi()
info = api.dataset_info(DATASET_ID, revision=REVISION, timeout=60)
assert info.sha == REVISION
raw_files = {s:sorted(f.rfilename for f in info.siblings
    if f.rfilename.startswith(f"data/{s}-") and f.rfilename.endswith(".parquet"))
    for s in ["train","test"]}
assert all(raw_files.values()), "Official Parquet split files were not found."
fs = HfFileSystem()
schema_rows = []
for split in ["train","test"]:
    remote = f"datasets/{DATASET_ID}@{REVISION}/{raw_files[split][0]}"
    # Read a Parquet footer, not HTML rows.
    with fs.open(remote,"rb",block_size=65536) as handle:
        schema = pq.ParquetFile(handle).schema_arrow
    assert set(schema.names) == set(EXPECTED_FIELDS), "Unexpected schema; stop."
    schema_rows += [{"split":split,"field":f.name,"arrow_type":str(f.type)} for f in schema]
SCHEMA_TABLE = pd.DataFrame(schema_rows)
SCHEMA_TABLE.to_csv(ROOT/"tables/source_schema.csv",index=False)
save_json("manifests/source_files.json", raw_files)
def stream_metadata(split):
    return load_dataset(DATASET_ID, name="default", revision=REVISION,
        split=split, streaming=True, columns=READ_COLUMNS, batch_size=BATCH_SIZE)
probe = next(iter(stream_metadata("train")))
assert set(probe) == set(READ_COLUMNS)
print("Pinned revision:", REVISION)
print("Shard counts:", {s:len(v) for s,v in raw_files.items()})
print("Projected fields:", list(probe), "| raw label:", probe["label"])
display(SCHEMA_TABLE)
del probe


In [ ]:
# Stream only four small columns. Persist in chunks; never collect raw HTML.
META_SCHEMA = pa.schema([("sha256",pa.string()),("url",pa.string()),
    ("label",pa.string()),("date",pa.date32())])
for split in ["train","test"]:
    start = time.perf_counter()
    source = stream_metadata(split)
    if RUN_MODE == "SMOKE":
        source = source.take(SMOKE_ROWS)
    tmp = metadata_path(split).with_suffix(".partial")
    count = 0
    with pq.ParquetWriter(tmp,META_SCHEMA,compression="zstd") as writer:
        buffer = []
        for row in source:
            buffer.append({name:row[name] for name in READ_COLUMNS})
            if len(buffer)==BATCH_SIZE:
                writer.write_table(pa.Table.from_pylist(buffer,schema=META_SCHEMA))
                count += len(buffer)
                buffer.clear()
                if count % (BATCH_SIZE*25)==0:
                    print(split,count,"metadata rows saved")
        if buffer:
            writer.write_table(pa.Table.from_pylist(buffer,schema=META_SCHEMA))
            count += len(buffer)
    assert count > 0
    tmp.replace(metadata_path(split))
    RUNTIMES.append({"stage":"metadata_stream","split":split,
                     "rows":count,"seconds":time.perf_counter()-start})

META, audit, missing, date_rows = {}, [], [], []
for split in ["train","test"]:
    df = pd.read_parquet(metadata_path(split))
    for column in READ_COLUMNS:
        bad = df[column].isna()
        if column != "date":
            bad = bad | df[column].astype("string").str.strip().eq("").fillna(False)
        missing.append({"split":split,"field":column,
                        "missing_n":int(bad.sum()),"missing_pct":100*bad.mean()})
    assert df["label"].isin(LABEL_MAP).all(), "Unknown/missing labels; stop."
    assert df["sha256"].astype("string").str.fullmatch(r"[0-9a-fA-F]{64}").fillna(False).all()
    df["date"] = pd.to_datetime(df["date"],errors="coerce")
    assert df["date"].notna().all(), "Invalid dates; no temporal split is allowed."
    df["label"] = df["label"].map(LABEL_MAP).astype("int8")
    df["source_row"] = np.arange(len(df),dtype="int64")
    df["record_id"] = split + ":" + df["source_row"].astype(str)
    META[split] = df
    duplicate_sha = int(df["sha256"].duplicated().sum())
    audit.append({"split":split,"rows":len(df),"reported_rows":REPORTED_ROWS[split],
        "benign":int((df.label==0).sum()),"phish":int((df.label==1).sum()),
        "date_min":str(df.date.min().date()),"date_max":str(df.date.max().date()),
        "duplicate_sha256":duplicate_sha,
        "duplicate_raw_url":int(df.url.dropna().duplicated().sum())})
    for label,g in df.groupby("label"):
        date_rows.append({"split":split,"label":int(label),"rows":len(g),
            "date_min":str(g.date.min().date()),"date_max":str(g.date.max().date())})
    if RUN_MODE == "FULL" and len(df) != REPORTED_ROWS[split]:
        BLOCKERS.append(f"{split}: measured row count differs from published total")
    if duplicate_sha:
        BLOCKERS.append(f"{split}: repeated sha256 requires identity/duplicate review")
SHA_OVERLAP = len(set(META["train"].sha256) & set(META["test"].sha256))
if SHA_OVERLAP:
    BLOCKERS.append("Exact sha256 overlap across official splits")
TEMPORAL_AUDIT = []
for label in [0,1]:
    a = META["train"].loc[lambda x:x.label==label,"date"]
    b = META["test"].loc[lambda x:x.label==label,"date"]
    if len(a) and len(b):
        relation = "strictly_later" if a.max()<b.min() else (
            "same_calendar_day_boundary" if a.max()==b.min() else "date_overlap")
        TEMPORAL_AUDIT.append({"label":label,"train_max":str(a.max().date()),
            "test_min":str(b.min().date()),"relation":relation})
        if RUN_MODE=="FULL" and relation=="date_overlap":
            BLOCKERS.append(f"Official class {label}: unexpected date overlap")
GLOBAL_LATER = bool(META["train"].date.max() < META["test"].date.min())
AUDIT = pd.DataFrame(audit)
MISSING = pd.DataFrame(missing)
DATE_RANGES = pd.DataFrame(date_rows)
# Date-only, whole-day, approximately 80/20 split; never optimise with scores.
days = META["train"].groupby("date").size().sort_index()
assert len(days)>=2, "Insufficient collection days even for a split preview."
cutoff = (days.cumsum().iloc[:-1]/days.sum()-0.80).abs().idxmin()
META["train"]["development_partition"] = np.where(
    META["train"].date<=cutoff,"internal_train","internal_validation")
SPLIT_SUMMARY = META["train"].groupby("development_partition").agg(
    rows=("label","size"),phish=("label","sum"),
    date_min=("date","min"),date_max=("date","max")).reset_index()
SPLIT_SUMMARY["benign"] = SPLIT_SUMMARY.rows-SPLIT_SUMMARY.phish
if RUN_MODE=="FULL" and ((SPLIT_SUMMARY.phish==0)|(SPLIT_SUMMARY.benign==0)).any():
    BLOCKERS.append("An internal temporal partition has only one class")
# No feature or label enters cutoff selection; labels only diagnose feasibility.
save_json("manifests/temporal_split_preflight.json",{
    "mode":RUN_MODE,"revision":REVISION,"cutoff_inclusive":str(cutoff.date()),
    "rule":"whole-day cumulative row count nearest 80%; earlier tie wins",
    "h08_decision_partition":"internal_train","official_test_unchanged":True})
for name,table in [("data_audit_summary",AUDIT),("required_field_missingness",MISSING),
    ("date_ranges_by_class",DATE_RANGES),("temporal_split_preflight",SPLIT_SUMMARY),
    ("official_temporal_audit",pd.DataFrame(TEMPORAL_AUDIT))]:
    table.to_csv(ROOT/"tables"/f"{name}.csv",index=False)
print("Exact cross-split SHA overlap:",SHA_OVERLAP)
print("One global strictly-later boundary:",GLOBAL_LATER)
print("HTML missingness: NOT MEASURED here; audit in Notebook 2.")
print("Published test counts sum to 168136, not 168060; use measured counts below.")
display(AUDIT); display(DATE_RANGES); display(SPLIT_SUMMARY)
print("Issues requiring review before modelling:", BLOCKERS)


In [ ]:
# Dataset description uses official-train metadata only.
t = META["train"]
counts = t.label.value_counts().reindex([0,1],fill_value=0)
fig,ax = plt.subplots(figsize=(5,3.4))
ax.bar(["Benign","Phishing"],counts.values,color=[COLORS[0],COLORS[1]])
ax.set(ylabel="Webpages",title=f"Official train class counts ({RUN_MODE})")
save_figure("F01_dataset_class_counts.png",fig)
monthly = t.assign(month=t.date.dt.to_period("M").astype(str)).groupby(
    ["month","label"]).size().unstack(fill_value=0).reindex(columns=[0,1],fill_value=0)
fig,ax = plt.subplots(figsize=(9,3.5))
monthly.rename(columns={0:"Benign",1:"Phishing"}).plot(
    kind="bar",stacked=True,ax=ax,color=[COLORS[0],COLORS[1]])
ax.set(xlabel="Collection month",ylabel="Webpages",title=f"Official train timeline ({RUN_MODE})")
save_figure("F02_collection_timeline_by_month.png",fig)
fig,ax = plt.subplots(figsize=(9,3.5))
ax.plot(monthly.index,100*monthly[1]/monthly.sum(axis=1),marker="o",color=COLORS[1])
ax.set(ylim=(0,100),xlabel="Collection month",ylabel="Phishing (%)",
       title=f"Official train monthly prevalence ({RUN_MODE})")
ax.tick_params(axis="x",rotation=60)
save_figure("F03_monthly_phishing_prevalence.png",fig)
m = MISSING.query("split=='train'")
fig,ax = plt.subplots(figsize=(6,3.5))
ax.bar(m.field,m.missing_pct,color=COLORS[0])
ax.set(ylim=(0,max(1,float(m.missing_pct.max())*1.2)),ylabel="Missing or blank (%)",
       title=f"Loaded fields only; HTML deferred ({RUN_MODE})")
save_figure("F04_required_field_missingness.png",fig)


## The fixed URL feature dictionary

Counts use the original URL string, with no percent-decoding or inserted scheme.
The parsing copy alone receives a scheme when absent. Digits mean ASCII `0–9`.

| ID | Feature | Operational definition | Source basis |
|---|---|---|---|
| U01 | url_length | Original character count | A, B, C |
| U02 | hostname_length | Parsed hostname characters, excluding credentials and port | B, C |
| U03 | path_length | Encoded path characters, including slashes | A, C |
| U04 | query_length | Encoded query characters, excluding `?` | C |
| U05 | num_dots | Dots in original URL | A, B, C |
| U06 | num_hyphens | Hyphens in original URL | B, C |
| U07 | num_digits | ASCII digits in original URL | A, C |
| U08 | digit_ratio | U07/U01; zero for empty input | A (fraction adaptation), B |
| U09 | num_subdomains | Labels before ICANN registrable domain; IP = 0 | A, B, C; PSL refinement |
| U10 | path_depth | Nonempty `/`-separated path segments | C |
| U11 | num_query_params | `parse_qsl`, keep blanks and repeated keys, separator `&` | A, C; explicit parsing rule |
| U12 | has_ip_host | `ipaddress` recognises IPv4 or IPv6 hostname | A, B, C |
| U13 | has_punycode | Any original hostname label starts with `xn--` | B |
| U14 | has_at_symbol | Literal `@` anywhere in original URL | B, C |
| U15 | url_entropy | Shannon character entropy, log base 2 | A |

A: Potpelwar, Kulkarni and Waghmare (2025), *LegitPhish*, Table 1.
https://doi.org/10.1016/j.dib.2025.111972

B: Yi, Omotosho and Balogun (2026), Table 3.
https://doi.org/10.1002/spy2.70175

C: Almousa et al. (2022), Table 6 and Appendix Table A1.
https://doi.org/10.1002/spy2.256

These sources document feature families; our exact parsing rules are declared
operationalisations, not claims of byte-for-byte reproduction. No other dataset is loaded.

PSL policy: tldextract 5.3.0 bundled snapshot; ICANN section only;
private suffixes excluded. Thus different `blogspot.com` tenants share one
registered domain: a declared limitation. Notebook 2 must reuse this policy.

Malformed inputs remain in the table. Raw-string features remain computable;
unavailable structural values are NaN, accompanied by nonpredictive diagnostics.
Unknown public suffixes give NaN for subdomain count. Notebook 3 must fit any
imputer only on internal training rows. No extra diagnostic becomes a predictor.


In [ ]:
FEATURES = ["url_length","hostname_length","path_length","query_length",
    "num_dots","num_hyphens","num_digits","digit_ratio","num_subdomains",
    "path_depth","num_query_params","has_ip_host","has_punycode",
    "has_at_symbol","url_entropy"]
STRUCTURAL = ["hostname_length","path_length","query_length","num_subdomains",
              "path_depth","num_query_params","has_ip_host","has_punycode"]
PSL_BYTES = resources.files("tldextract").joinpath(".tld_set_snapshot").read_bytes()
(ROOT/"manifests/public_suffix_list.dat").write_bytes(PSL_BYTES)
PSL_SHA256 = hashlib.sha256(PSL_BYTES).hexdigest()
DOMAIN_EXTRACTOR = tldextract.TLDExtract(
    suffix_list_urls=(),cache_dir=None,include_psl_private_domains=False)
def entropy(text):
    if not text: return 0.0
    return -sum((n/len(text))*math.log2(n/len(text)) for n in Counter(text).values())
def parse_url(raw):
    text = raw if isinstance(raw,str) else ""
    out = {"raw":text,"parsed":None,"host":"","registered_domain":"",
           "subdomain_count":np.nan,"ip":False,"parse_failed":True,
           "unknown_suffix":False}
    s = text.strip()
    if not s or any(ord(c)<32 or c.isspace() for c in s): return out
    # Explicit schemes, protocol-relative URLs, and scheme-less host:port URLs.
    prefix = re.match(r"^([A-Za-z][A-Za-z0-9+.-]*):",s)
    explicit = bool(prefix and ("://" in s or "." not in prefix.group(1)))
    candidate = ("http:"+s if s.startswith("//") else s if explicit else "http://"+s)
    try:
        p = urlsplit(candidate)
        if p.scheme.lower() not in {"http","https"} or not p.hostname: return out
        port = p.port                 # Validate invalid/out-of-range ports too.
        host = p.hostname.lower().rstrip(".")
        try:
            ipaddress.ip_address(host)
            is_ip = True
        except ValueError:
            is_ip = False
        if not is_ip:
            ascii_host = host.encode("idna").decode("ascii").lower()
            labels = ascii_host.split(".")
            if len(ascii_host)>253 or any(not re.fullmatch(
                r"[a-z0-9_](?:[a-z0-9_-]{0,61}[a-z0-9_])?",x) for x in labels):
                return out
            ext = DOMAIN_EXTRACTOR(ascii_host)
            domain = ext.top_domain_under_public_suffix
            nsub = len(ext.subdomain.split(".")) if domain and ext.subdomain else (
                0 if domain else np.nan)
        else:
            domain,nsub = "",0
        out.update(parsed=p,host=host,registered_domain=domain,
            subdomain_count=nsub,ip=is_ip,parse_failed=False,
            unknown_suffix=(not is_ip and not domain))
    except (ValueError,UnicodeError):
        pass
    return out
print("15 features; offline PSL snapshot SHA256:",PSL_SHA256)


In [ ]:
assert parse_url("https://secure.example.co.uk/a")["registered_domain"]=="example.co.uk"
assert parse_url("https://secure.example.co.uk/a")["subdomain_count"]==1
assert parse_url("example.com:8080/a")["host"]=="example.com"
assert parse_url("//example.com/a")["host"]=="example.com"
assert parse_url("http://192.0.2.1:8080/a")["ip"]
assert parse_url("https://[2001:db8::1]/")["ip"]
assert parse_url("https://user@example.com/a")["host"]=="example.com"
assert parse_url("https://xn--bcher-kva.de/")["registered_domain"]=="xn--bcher-kva.de"
assert parse_url("https://alice.blogspot.com")["registered_domain"]=="blogspot.com"
assert parse_url("http://[broken")["parse_failed"]
assert parse_url("https://example.com:bad/")["parse_failed"]
assert parse_url("javascript:alert(1)")["parse_failed"]
assert parse_url("not a URL")["parse_failed"]
assert parse_url(None)["parse_failed"]
assert parse_url("https://example.invalid")["unknown_suffix"]
assert entropy("")==0 and entropy("aaaa")==0 and abs(entropy("ab")-1)<1e-12
assert len(parse_qsl("a=1&a=2&b=&c",keep_blank_values=True,separator="&"))==4
assert len(parse_url("https://example.com/?q="+"x"*10000)["parsed"].query)==10002
print("18 parsing/entropy edge checks passed. No URLs were visited.")


In [ ]:
def extract_url_features(raw):
    info = parse_url(raw)
    text = info["raw"]
    digits = sum("0"<=c<="9" for c in text)
    f = {name:np.nan for name in FEATURES}
    f.update(url_length=len(text),num_dots=text.count("."),
        num_hyphens=text.count("-"),num_digits=digits,
        digit_ratio=digits/len(text) if text else 0.0,
        has_at_symbol=int("@" in text),url_entropy=entropy(text))
    if not info["parse_failed"]:
        p = info["parsed"]
        f.update(hostname_length=len(p.hostname),path_length=len(p.path),
            query_length=len(p.query),num_subdomains=info["subdomain_count"],
            path_depth=sum(bool(x) for x in p.path.split("/")),
            num_query_params=len(parse_qsl(p.query,keep_blank_values=True,separator="&")),
            has_ip_host=int(info["ip"]),
            has_punycode=int(any(x.startswith("xn--") for x in p.hostname.lower().split("."))))
    diagnostics = {"url_parse_failed":info["parse_failed"],
        "unknown_public_suffix":info["unknown_suffix"],
        "registered_domain":info["registered_domain"]}
    return {k:f[k] for k in FEATURES}, diagnostics
f,_ = extract_url_features("https://a.example.co.uk/a//b/?x=1&x=&y")
assert f["path_depth"]==2 and f["num_query_params"]==3 and f["num_subdomains"]==1
f,_ = extract_url_features("https://example.com/%2F?a=%40")
assert f["path_length"]==4 and f["has_at_symbol"]==0
f,_ = extract_url_features("https://xn--bcher-kva.de/")
assert f["has_punycode"]==1
f,d = extract_url_features("http://[broken")
assert d["url_parse_failed"] and math.isnan(f["hostname_length"])
f,_ = extract_url_features("http://192.0.2.1/")
assert f["has_ip_host"]==1 and f["num_subdomains"]==0
preview = pd.DataFrame([extract_url_features(u)[0] for u in META["train"].url.head(10)])
assert list(preview)==FEATURES and len(FEATURES)==15
print("Feature checks passed. Preview has exactly 15 predictive columns.")
display(preview)


In [ ]:
# One extractor, used without any change for both official splits.
def extract_split(split):
    start = time.perf_counter()
    tmp = feature_path(split).with_suffix(".partial")
    frame = META[split]
    fields = [("sha256",pa.string()),("url",pa.string()),("label",pa.int8()),
        ("date",pa.timestamp("ns")),("source_row",pa.int64()),("record_id",pa.string())]
    if split=="train": fields.append(("development_partition",pa.string()))
    fields += [(name,pa.float64()) for name in FEATURES]
    fields += [("url_parse_failed",pa.bool_()),("unknown_public_suffix",pa.bool_()),
               ("registered_domain",pa.string())]
    output_schema = pa.schema(fields)
    writer = None
    extracted = 0
    try:
        for lo in range(0,len(frame),BATCH_SIZE):
            block = frame.iloc[lo:lo+BATCH_SIZE].copy()
            output = [extract_url_features(u) for u in block.url]
            numeric = pd.DataFrame([x[0] for x in output],columns=FEATURES).astype("float64")
            diag = pd.DataFrame([x[1] for x in output])
            assert not np.isinf(numeric.to_numpy()).any()
            assert numeric["digit_ratio"].between(0,1).all()
            block = pd.concat([block.reset_index(drop=True),numeric,diag],axis=1)
            # Identical schema across batches: preserve metadata but never HTML.
            table = pa.Table.from_pandas(block,schema=output_schema,preserve_index=False)
            if writer is None: writer=pq.ParquetWriter(tmp,table.schema,compression="zstd")
            writer.write_table(table)
            extracted += len(block)
            if extracted % (BATCH_SIZE*25)==0: print(split,extracted,"URL rows saved")
    finally:
        if writer is not None: writer.close()
    assert extracted==len(frame)
    tmp.replace(feature_path(split))
    seconds = time.perf_counter()-start
    RUNTIMES.append({"stage":"url_extract_and_write","split":split,
                     "rows":extracted,"seconds":seconds})
    print(split,extracted,"rows;",round(seconds,2),"seconds including Parquet writes")
extract_split("train")


In [ ]:
extract_split("test")
print("Official-test features saved mechanically. No model or performance metric exists.")


In [ ]:
train_features = pd.read_parquet(feature_path("train"))
# EDA is confined to the EARLIER internal training partition.
eda = train_features.query("development_partition=='internal_train'")
STATS = eda.groupby("label")[FEATURES].describe().T.reset_index()
CORR = eda[FEATURES].corr(method="spearman")
FEATURE_MISSING = pd.DataFrame({"feature":FEATURES,
    "missing_n":eda[FEATURES].isna().sum().values,
    "missing_pct":100*eda[FEATURES].isna().mean().values})
STATS.to_csv(ROOT/"tables/url_feature_descriptive_stats_train.csv",index=False)
CORR.to_csv(ROOT/"tables/url_feature_correlations_train.csv")
FEATURE_MISSING.to_csv(ROOT/"tables/url_feature_missingness_internal_train.csv",index=False)
for feature,filename,xlabel in [
    ("url_length","F05_url_length_by_class.png","URL length (characters; log scale)"),
    ("url_entropy","F06_url_entropy_by_class.png","Shannon entropy (bits/character)")]:
    fig,ax = plt.subplots(figsize=(6.5,3.7))
    for label,name in [(0,"Benign"),(1,"Phishing")]:
        values=np.sort(eda.loc[eda.label==label,feature].dropna().to_numpy())
        if len(values): ax.plot(values,np.arange(1,len(values)+1)/len(values),
                               label=name,color=COLORS[label])
    if feature=="url_length": ax.set_xscale("symlog",linthresh=1)
    ax.set(xlabel=xlabel,ylabel="Cumulative fraction",ylim=(0,1.01),
           title=f"Earlier internal train only ({RUN_MODE})")
    ax.legend()
    save_figure(filename,fig)
fig,ax=plt.subplots(figsize=(10,8))
im=ax.imshow(CORR.to_numpy(),cmap="RdBu_r",vmin=-1,vmax=1)
ax.set_xticks(range(15),FEATURES,rotation=90)
ax.set_yticks(range(15),FEATURES)
ax.set_title(f"Spearman correlation: earlier internal train ({RUN_MODE})")
fig.colorbar(im,ax=ax,label="Spearman correlation")
save_figure("F07_url_feature_correlation_heatmap.png",fig)
print("Blank correlation cells indicate constant/all-missing features, not zero correlation.")
print("No features were added or removed from these figures.")


In [ ]:
definitions = [
("Original URL characters","A; B; C"),("Parsed hostname characters; no port/userinfo","B; C"),
("Encoded path characters","A; C"),("Encoded query characters without ?","C"),
("Original URL dot count","A; B; C"),("Original URL hyphen count","B; C"),
("ASCII digits 0-9 in original URL","A; C"),("ASCII digits / original length; empty=0","A adapted; B"),
("Labels before ICANN registrable domain; IP=0; unknown=NaN","A; B; C; PSL refinement"),
("Nonempty path segments","C"),("Parsed query pairs; repeated/blank retained; & separator","A; C adapted"),
("Parsed host recognised as IPv4/IPv6","A; B; C"),("Original hostname label begins xn--","B"),
("Literal @ anywhere in original URL","B; C"),("Shannon entropy, log2, original characters","A")]
DICTIONARY = pd.DataFrame([{"id":f"U{i:02d}","feature":name,"definition":definition,
    "source":source,"predictor":True} for i,(name,(definition,source)) in
    enumerate(zip(FEATURES,definitions),1)])
SOURCES = pd.DataFrame([
    {"key":"A","citation":"Potpelwar, Kulkarni and Waghmare (2025), Table 1",
     "url":"https://doi.org/10.1016/j.dib.2025.111972"},
    {"key":"B","citation":"Yi, Omotosho and Balogun (2026), Table 3",
     "url":"https://doi.org/10.1002/spy2.70175"},
    {"key":"C","citation":"Almousa et al. (2022), Table 6 and Appendix A1",
     "url":"https://doi.org/10.1002/spy2.256"}])
DICTIONARY.to_csv(ROOT/"tables/url_feature_dictionary.csv",index=False)
SOURCES.to_csv(ROOT/"tables/feature_sources.csv",index=False)
# Compact advisor preview: safe IDs + numeric features, no clickable live URLs.
PREVIEW = train_features[["record_id","label"]+FEATURES].head(100)
PREVIEW.to_csv(ROOT/"tables/url_feature_table_preview_train.csv",index=False)
QC_ROWS=[]
for split in ["train","test"]:
    q=pd.read_parquet(feature_path(split),columns=["url_parse_failed","unknown_public_suffix"])
    QC_ROWS.append({"split":split,"rows":len(q),
        "url_parse_failures":int(q.url_parse_failed.sum()),
        "unknown_public_suffix":int(q.unknown_public_suffix.sum())})
QC=pd.DataFrame(QC_ROWS)
QC.to_csv(ROOT/"tables/url_parser_qc.csv",index=False)
pd.DataFrame(RUNTIMES).to_csv(ROOT/"logs/nb01_runtime.csv",index=False)
workbook=ROOT/"tables/NB01_Data_Audit_and_URL_Features.xlsx"
with pd.ExcelWriter(workbook,engine="xlsxwriter",
    engine_kwargs={"options":{"strings_to_urls":False,"strings_to_formulas":False}}) as xw:
    sheets={"Data audit":AUDIT,"Schema":SCHEMA_TABLE,"Date ranges":DATE_RANGES,
        "Internal split":SPLIT_SUMMARY,"Missing fields":MISSING,
        "Feature dictionary":DICTIONARY,"Sources":SOURCES,"Feature preview":PREVIEW,
        "URL summary":STATS,"Correlations":CORR.reset_index(),
        "Feature missingness":FEATURE_MISSING,"Parser QC":QC,"Runtime":pd.DataFrame(RUNTIMES)}
    for sheet,table in sheets.items():
        table.to_excel(xw,sheet_name=sheet,index=False)
        ws=xw.sheets[sheet]
        ws.freeze_panes(1,0)
        ws.autofilter(0,0,len(table),len(table.columns)-1)
        ws.set_column(0,len(table.columns)-1,21)
print("Excel summary:",workbook.name)
print("Complete numeric feature tables are Parquet; Excel contains a 100-row preview.")


In [ ]:
# Verify persisted features in small batches, including row alignment and NaN policy.
for split in ["train","test"]:
    pf=pq.ParquetFile(feature_path(split))
    assert pf.metadata.num_rows==len(META[split])
    offset=0
    for batch in pf.iter_batches(batch_size=BATCH_SIZE):
        b=batch.to_pandas()
        expected=META[split].iloc[offset:offset+len(b)]
        assert b.record_id.tolist()==expected.record_id.tolist()
        assert b.sha256.tolist()==expected.sha256.tolist()
        assert b.label.tolist()==expected.label.tolist()
        assert list(b[FEATURES])==FEATURES
        x=b[FEATURES]
        assert not np.isinf(x.to_numpy()).any()
        assert (x.fillna(0)>=0).all().all()
        for name in FEATURES:
            allowed = b.url_parse_failed if name in STRUCTURAL else pd.Series(False,index=b.index)
            if name=="num_subdomains": allowed=allowed|b.unknown_public_suffix
            assert (~x[name].isna() | allowed).all(), f"Unexpected missing {name}"
        for name in ["has_ip_host","has_punycode","has_at_symbol"]:
            assert x[name].dropna().isin([0,1]).all()
        offset+=len(b)
save_json("manifests/label_mapping.json",LABEL_MAP)
pd.DataFrame(list(versions.items()),columns=["package","version"]).to_csv(
    ROOT/"manifests/package_versions_nb01.csv",index=False)
save_json("manifests/url_feature_names.json",FEATURES)
manifest={"notebook":"01_data_audit_and_url_features.ipynb","mode":RUN_MODE,
    "dataset_id":DATASET_ID,"dataset_version":DATASET_VERSION,"revision":REVISION,
    "rows":{s:len(d) for s,d in META.items()},"features":FEATURES,"seed":SEED,
    "psl":{"package":"tldextract==5.3.0","sha256":PSL_SHA256,"private_domains":False},
    "internal_cutoff_inclusive":str(cutoff.date()),"official_test_evaluated":False,
    "html_missingness":"deferred to Notebook 2", "sha_overlap":SHA_OVERLAP,
    "global_strict_temporal_boundary":GLOBAL_LATER,"class_temporal_audit":TEMPORAL_AUDIT,
    "parser_policy":"retain malformed rows; structural NaNs; diagnostics excluded",
    "blockers":BLOCKERS,"python":platform.python_version(),"packages":versions,
    "ready_for_nb02":RUN_MODE=="FULL" and not BLOCKERS}
manifest["artifacts"]={str(p.relative_to(ROOT)):{"bytes":p.stat().st_size,"sha256":sha_file(p)}
    for p in sorted(ROOT.rglob("*")) if p.is_file() and p.suffix!=".partial" and p.name!="manifest_nb01.json"}
save_json("manifests/manifest_nb01.json",manifest)
print("MODE:",RUN_MODE,"| Rows:",manifest["rows"])
print("Predictors:",len(FEATURES),"| Official test evaluated: NO")
print("Ready for Notebook 2:",manifest["ready_for_nb02"])
if BLOCKERS:
    print("STOP BEFORE MODELLING: inspect",BLOCKERS)
elif RUN_MODE=="SMOKE":
    print("SMOKE COMPLETE. Change Cell 3 to FULL; run all 15 cells again.")
else:
    print("FULL COMPLETE. Save Version -> Save & Run All, with outputs saved.")
    print("Attach this version's full output folder to Notebook 2.")
